# Pemodelan Jasa Lingkungan Hidup (JLH) Sumatera Barat

**Penyusun:** Direktorat Penyelenggaraan Sumber Daya Alam Berkelanjutan  

Notebook ini digunakan untuk menghitung **6 Indikator Jasa Lingkungan Hidup (JLH)** menggunakan data:
1. **KBA (Kawasan Berhutan Alami)** - 6 file raster COG (0.5 meter)
2. **KVA (Kawasan Vegetasi Alami)** - 6 file raster COG (0.5 meter)
3. **Penutupan Lahan (PL)** - ~2.300+ file raster grid tile (0.5 meter)

---
### Rumus Perhitungan:
$$\text{JLH} = (\text{Skor KBA} \times W_{\text{KBA}}) + (\text{Skor KVA} \times W_{\text{KVA}}) + (\text{Skor PL} \times W_{\text{PL}})$$

### 6 Indikator JLH & Bobot:

| Indikator JLH | Bobot KBA (W_KBA) | Bobot KVA (W_KVA) | Bobot PL (W_PL) |
| :--- | :---: | :---: | :---: |
| **PGA** (Pengaturan Air) | 0.28 | 0.12 | 0.60 |
| **PHK** (Pendukung Habitat & Kehati) | 0.24 | 0.16 | 0.60 |
| **PPK** (Penyerap & Penyimpan Karbon) | 0.20 | 0.20 | 0.60 |
| **PYA** (Penyedia Air) | 0.32 | 0.08 | 0.60 |
| **PGN** (Penyedia Pangan - Sumatera) | 0.28 | 0.12 | 0.60 |
| **PKU** (Pengatur Kualitas Udara) | 0.15 | 0.25 | 0.60 |

---
### Skor Penutupan Lahan (PL):

| Value | Kelas | PL PGA | PL PHK | PL PPK | PL PYA | PL PGN (Sumatera) | PL PKU |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **1** | Water | 5 | 1 | 1 | 5 | 4 | 5 |
| **2** | Natural Forest | 5 | 5 | 5 | 3 | 4 | 5 |
| **4** | Flooded Vegetation | 4 | 3 | 3 | 3 | 4 | 5 |
| **5** | Crops | 3 | 1 | 1 | 3 | 4 | 3 |
| **7** | Development Space | 2 | 1 | 1 | 2 | 1 | 2 |
| **8** | Bare Ground | 1 | 1 | 1 | 1 | 1 | 1 |
| **11** | Rangeland | 3 | 2 | 2 | 1 | 2 | 3 |
| **12** | Plantation | 3 | 2 | 2 | 2 | 3 | 4 |
| **13** | Uncertain Trees | 3 | 3 | 2 | 2 | 3 | 3 |
| **14** | Road | 1 | 1 | 1 | 1 | 1 | 1 |
| **15** | Building | 1 | 1 | 1 | 2 | 1 | 1 |

> **Catatan Arsitektur:** Pemrosesan dilakukan **per-grid tile** menggunakan teknik *windowed reading* pada file COG KBA/KVA. Hasil per-grid disimpan sebagai 1 file GeoTIFF 6-band terkompresi DEFLATE (ringan & anti-OOM/memory crash), kemudian digabungkan secara virtual menjadi `.vrt` untuk QGIS.


In [ ]:
# Langkah 1: Instalasi pustaka geospasial yang dibutuhkan
!pip install -q rasterio tqdm

### Langkah 2: Mount Google Drive & Konfigurasi Parameter Bobot & Skor PL

In [ ]:
import os
import glob
import gc
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from tqdm.notebook import tqdm
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definisi Path Folder
PATH_PL = '/content/drive/MyDrive/class'
PATH_KBA = '/content/drive/MyDrive/Data Lain/ANALISA/JLH/01_raster/01_kba'
PATH_KVA = '/content/drive/MyDrive/Data Lain/ANALISA/JLH/01_raster/02_kva'
PATH_OUTPUT = '/content/drive/MyDrive/Data Lain/ANALISA/JLH/02_output_jlh_grids'

os.makedirs(PATH_OUTPUT, exist_ok=True)

# 3. Konfigurasi 6 Indikator JLH & Bobot
JLH_CONFIG = {
    'pga': {'name': 'Pengaturan Air',             'w_kba': 0.28, 'w_kva': 0.12, 'w_pl': 0.60},
    'phk': {'name': 'Pendukung Habitat & Kehati', 'w_kba': 0.24, 'w_kva': 0.16, 'w_pl': 0.60},
    'ppk': {'name': 'Penyerap & Simpan Karbon',   'w_kba': 0.20, 'w_kva': 0.20, 'w_pl': 0.60},
    'pya': {'name': 'Penyedia Air',               'w_kba': 0.32, 'w_kva': 0.08, 'w_pl': 0.60},
    'pgn': {'name': 'Penyedia Pangan (Sumatera)', 'w_kba': 0.28, 'w_kva': 0.12, 'w_pl': 0.60},
    'pku': {'name': 'Pengatur Kualitas Udara',    'w_kba': 0.15, 'w_kva': 0.25, 'w_pl': 0.60}
}

# 4. Lookup Skor Penutupan Lahan (PL)
# Value: 1:Water, 2:Natural Forest, 4:Flooded Veg, 5:Crops, 7:Dev Space,
#        8:Bare Ground, 11:Rangeland, 12:Plantation, 13:Uncertain Trees, 14:Road, 15:Building
PL_LOOKUP = {
    1:  {'pga': 5, 'phk': 1, 'ppk': 1, 'pya': 5, 'pgn': 4, 'pku': 5},
    2:  {'pga': 5, 'phk': 5, 'ppk': 5, 'pya': 3, 'pgn': 4, 'pku': 5},
    4:  {'pga': 4, 'phk': 3, 'ppk': 3, 'pya': 3, 'pgn': 4, 'pku': 5},
    5:  {'pga': 3, 'phk': 1, 'ppk': 1, 'pya': 3, 'pgn': 4, 'pku': 3},
    7:  {'pga': 2, 'phk': 1, 'ppk': 1, 'pya': 2, 'pgn': 1, 'pku': 2},
    8:  {'pga': 1, 'phk': 1, 'ppk': 1, 'pya': 1, 'pgn': 1, 'pku': 1},
    11: {'pga': 3, 'phk': 2, 'ppk': 2, 'pya': 1, 'pgn': 2, 'pku': 3},
    12: {'pga': 3, 'phk': 2, 'ppk': 2, 'pya': 2, 'pgn': 3, 'pku': 4},
    13: {'pga': 3, 'phk': 3, 'ppk': 2, 'pya': 2, 'pgn': 3, 'pku': 3},
    14: {'pga': 1, 'phk': 1, 'ppk': 1, 'pya': 1, 'pgn': 1, 'pku': 1},
    15: {'pga': 1, 'phk': 1, 'ppk': 1, 'pya': 2, 'pgn': 1, 'pku': 1}
}

# Buat array re-map 1D NumPy (index 0 - 255) untuk operasi matriks super cepat
def create_remap_array(jlh_key):
    arr = np.zeros(256, dtype=np.float32)
    for c_val, scores in PL_LOOKUP.items():
        arr[c_val] = scores[jlh_key]
    return arr

PL_REMAP = {k: create_remap_array(k) for k in JLH_CONFIG.keys()}
print("✅ Inisialisasi parameter dan lookup table berhasil!")

### Langkah 3: Verifikasi File KBA & KVA
Cell ini akan memeriksa dan memetakan 6 file TIF di folder KBA dan 6 file TIF di folder KVA.

In [ ]:
def map_raster_files(folder_path):
    all_tifs = sorted(glob.glob(os.path.join(folder_path, "*.tif")))
    mapped = {}
    for key in JLH_CONFIG.keys():
        for path in all_tifs:
            fname = os.path.basename(path).lower()
            if key in fname:
                mapped[key] = path
                break
    return mapped, all_tifs

kba_files, all_kba = map_raster_files(PATH_KBA)
kva_files, all_kva = map_raster_files(PATH_KVA)

print("=" * 50)
print("📌 HASIL DETEKSI FILE KBA:")
for k, v in kba_files.items():
    print(f"  [{k.upper()}] -> {os.path.basename(v)}")
print(f"Total file ditemukan di folder KBA: {len(all_kba)}")

print()
print("📌 HASIL DETEKSI FILE KVA:")
for k, v in kva_files.items():
    print(f"  [{k.upper()}] -> {os.path.basename(v)}")
print(f"Total file ditemukan di folder KVA: {len(all_kva)}")
print("=" * 50)

if len(kba_files) < 6 or len(kva_files) < 6:
    print("⚠️ PERINGATAN: Ada indikator yang belum terpetakan otomatis!")
    print("Daftar semua file di folder KBA:", [os.path.basename(p) for p in all_kba])
    print("Daftar semua file di folder KVA:", [os.path.basename(p) for p in all_kva])
    print("Silakan sesuaikan nama file atau mapping manual jika nama file berbeda.")
else:
    print("✅ Semua 6 file KBA dan 6 file KVA berhasil terpetakan!")


### Langkah 4: Eksekusi Komputasi JLH Per-Grid (Multi-Band)
Cell ini akan:
1. Melakukan perulangan untuk setiap file grid tutupan lahan ().
2. Menggunakan **Windowed Reading** pada file COG KBA & KVA (hemat memori, anti-crash).
3. Menghitung rumus JLH untuk 6 indikator.
4. Menyimpan output per-grid sebagai **1 file GeoTIFF 6-Band** terkompresi DEFLATE.
5. Memiliki fitur **Auto-Resume** (bisa diulang kapan saja jika koneksi putus tanpa mengulang dari awal).

In [ ]:
# 1. Buka dataset COG KBA dan KVA (Read-only handles)
kba_ds = {k: rasterio.open(path) for k, path in kba_files.items()}
kva_ds = {k: rasterio.open(path) for k, path in kva_files.items()}

# 2. Ambil seluruh daftar file grid tutupan lahan
pl_grid_files = sorted(glob.glob(os.path.join(PATH_PL, "sumbar_50cm_*_multiscale_lulc.tif")))
print(f"Total grid tutupan lahan yang siap diproses: {len(pl_grid_files):,} file")

jlh_keys = list(JLH_CONFIG.keys())  # ['pga', 'phk', 'ppk', 'pya', 'pgn', 'pku']

# 3. Looping pemrosesan per-grid
success_count = 0
skip_count = 0

for pl_path in tqdm(pl_grid_files, desc="Memproses Grid JLH"):
    grid_fname = os.path.basename(pl_path)
    # Format nama output: sumbar_50cm_XXXX_JLH_6band.tif
    out_fname = grid_fname.replace("_multiscale_lulc.tif", "_JLH_6band.tif")
    out_path = os.path.join(PATH_OUTPUT, out_fname)

    # Fitur Auto-Resume: Lewati jika file hasil sudah ada
    if os.path.exists(out_path):
        skip_count += 1
        continue

    with rasterio.open(pl_path) as src_pl:
        pl_data = src_pl.read(1)
        h, w = src_pl.height, src_pl.width
        bounds = src_pl.bounds
        nodata_mask = (pl_data == 0) | (pl_data == 255)

        # Siapkan array penampung 6 band (float32)
        jlh_result = np.zeros((6, h, w), dtype=np.float32)

        for idx, key in enumerate(jlh_keys):
            cfg = JLH_CONFIG[key]

            # A. Re-map nilai tutupan lahan ke Skor PL
            skor_pl = PL_REMAP[key][pl_data]

            # B. Windowed Read KBA dari COG (dengan Boundary & Boundless Check)
            kba_b = kba_ds[key].bounds
            if (bounds.right <= kba_b.left or bounds.left >= kba_b.right or
                bounds.top <= kba_b.bottom or bounds.bottom >= kba_b.top):
                skor_kba = np.zeros((h, w), dtype=np.float32)
            else:
                win_kba = from_bounds(*bounds, transform=kba_ds[key].transform)
                skor_kba = kba_ds[key].read(1, window=win_kba, out_shape=(h, w), resampling=Resampling.nearest, boundless=True, fill_value=0)

            # C. Windowed Read KVA dari COG (dengan Boundary & Boundless Check)
            kva_b = kva_ds[key].bounds
            if (bounds.right <= kva_b.left or bounds.left >= kva_b.right or
                bounds.top <= kva_b.bottom or bounds.bottom >= kva_b.top):
                skor_kva = np.zeros((h, w), dtype=np.float32)
            else:
                win_kva = from_bounds(*bounds, transform=kva_ds[key].transform)
                skor_kva = kva_ds[key].read(1, window=win_kva, out_shape=(h, w), resampling=Resampling.nearest, boundless=True, fill_value=0)

            # D. Perhitungan Rumus JLH
            # jlh = (skor kba * w_kba) + (skor kva * w_kva) + (skor pl * w_pl)
            jlh_band = (skor_kba * cfg['w_kba']) + (skor_kva * cfg['w_kva']) + (skor_pl * cfg['w_pl'])
            jlh_band[nodata_mask] = 0.0
            jlh_result[idx] = jlh_band

        # E. Simpan Output Multi-band GeoTIFF dengan kompresi Deflate
        out_profile = src_pl.profile.copy()
        out_profile.update({
            'count': 6,
            'dtype': 'float32',
            'nodata': 0.0,
            'compress': 'deflate',
            'predictor': 3  # Optimal untuk kompresi data numerik float
        })

        with rasterio.open(out_path, 'w', **out_profile) as dst:
            dst.write(jlh_result)
            for b_idx, key in enumerate(jlh_keys, 1):
                dst.set_band_description(b_idx, f"JLH_{key.upper()}_{JLH_CONFIG[key]['name']}")

    success_count += 1
    del pl_data, jlh_result
    gc.collect()

# Tutup handles
for ds in kba_ds.values(): ds.close()
for ds in kva_ds.values(): ds.close()

print("=" * 50)
print("🎉 PEMROSESAN SELESAI!")
print(f"• Grid baru diproses : {success_count:,}")
print(f"• Grid dilewati (ada): {skip_count:,}")
print(f"• Lokasi hasil       : {PATH_OUTPUT}")
print("=" * 50)


### Langkah 5: Pembuatan Virtual Raster (.vrt) untuk QGIS / ArcGIS
Setelah seluruh grid selesai diproses, cell ini akan membuat **6 file VRT** (satu file untuk masing-masing indikator PGA, PHK, PPK, PYA, PGN, PKU).

File  ini ukurannya sangat kecil (<5 MB) tetapi saat ditarik ke QGIS akan **langsung tampil sebagai 1 peta mozaik utuh se-Sumatera Barat**!

In [ ]:
from osgeo import gdal

output_grids = sorted(glob.glob(os.path.join(PATH_OUTPUT, "*_JLH_6band.tif")))
vrt_dir = os.path.join(PATH_OUTPUT, "vrt_mosaic")
os.makedirs(vrt_dir, exist_ok=True)

print(f"Membuat Virtual Raster (VRT) dari {len(output_grids):,} file grid...")
print()

for band_idx, key in enumerate(jlh_keys, 1):
    vrt_path = os.path.join(vrt_dir, f"Sumbar_JLH_{key.upper()}_Mosaic.vrt")
    vrt_options = gdal.BuildVRTOptions(bandList=[band_idx])
    gdal.BuildVRT(vrt_path, output_grids, options=vrt_options)
    print(f"✅ VRT Terbuat: {os.path.basename(vrt_path)} (Band {band_idx}: {key.upper()})")

print()
print("🚀 SEMUA VRT SELESAI DIBUAT!")
print("Buka file .vrt di QGIS langsung dari folder:")
print(f"👉 {vrt_dir}")
